# Prompt sensitivity

**Session 3 · Track A · local Ollama**

Show that trivial wording changes move the numbers — so test on purpose.

In [1]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
from eval import load_cases, run_eval, print_report, exact
from utils import ask


### Worked example

Build trivial variants of one prompt - reordered examples, changed case, reworded header - and score them all on the same eval set. Report the spread.


In [2]:
# Worked example: trivial wording changes, identical eval set
cases = load_cases("../eval/datasets/sentiment.jsonl")
LABS = ["positive", "negative", "neutral"]
EXAMPLES = [("I love this", "positive"), ("crashed twice", "negative"), ("it is a phone", "neutral")]

def build(examples, header="Classify sentiment as positive, negative, or neutral. Reply with one lowercase word."):
    return "\n".join([header, ""] + [f'Text: "{t}" -> {l}' for t, l in examples] + [""])

def clf(fewshot):
    def f(text):
        out = ask(fewshot + f'Text: "{text}" -> ').strip().lower()
        return next((l for l in LABS if l in out), out)
    return f

variants = {
    "V1 baseline":  build(EXAMPLES),
    "V2 reordered": build(list(reversed(EXAMPLES))),
    "V3 lowercase": build(EXAMPLES).lower(),
    "V4 reworded":  build(EXAMPLES, header="classify sentiment - positive, negative or neutral - one lowercase word"),
}
scores = {name: run_eval(cases, clf(v), scorer=exact)["accuracy"] for name, v in variants.items()}
for name, acc in scores.items():
    print(f"  {name:13} {acc:.0%}")
print(f"\nspread: {max(scores.values()) - min(scores.values()):.0%}")


  V1 baseline   93%
  V2 reordered  87%
  V3 lowercase  87%
  V4 reworded   93%

spread: 7%


## Your turn - vary the example

1. Add two more trivial variants (extra whitespace, punctuation, a "Please").
2. Which variant wins? Re-run once more - is the winner stable or noise?
3. A prompt that only wins by luck is not done. What spread would you accept?


In [ ]:
# Your variation here - copy the worked example above and change ONE thing, then re-run
